In [20]:
from crewai import Agent, Task, Crew, LLM
from crewai.flow.flow import listen, start, or_, router
from crewai_tools import SerperDevTool
from crewai import Flow
from pydantic import BaseModel
from uuid import uuid4
import os

In [21]:
api_key = "langdb_NjVXcFI0SjZidmROcCtGa0ZKSVVVMVVYMnMwY2JjUGVmVy93elJjNEpOT3FNK0lQbWJ2aUZqWEZSb2M2TFZxQmozWUpOcy9kR0JIWS9TbmtOMktUVkdXem82bXJCOGMxeVJuVUdjazdFekh0QnpxZElTaFZOaVZJOHFhUTZrVGgzUUJXRlZTaUFqUS9Uc3pkY2Q2Y3BraXUwQVJ1WlcvNERWeWhMcTJ0KzVZZ0wzTHlyYXF4cW1UakFpc1ZTV05LRTVIREd3PT06QUFBQUFBQUFBQUFBQUFBQQ=="
url = "https://api.staging.langdb.ai"
os.environ["SERPER_API_KEY"] = "cb13349310f7b7a9984e1e6c44ea785120417ee1"

In [22]:
parent_id = str(uuid4())
headers1 = {"x-trace-id": parent_id}
llm1 = LLM(
    model="gpt-4o-mini",
    api_base=url,
    api_key=api_key,
    extra_headers=headers1,
)

In [23]:
movie_agent = Agent(
    role="Recommend popular movie specific to the genre",
    goal="Provide a list of movies based on user preferences",
    backstory="You are a cinephile, "
    "you recommend good movies to your friends, "
    "the movies should be of the same genre",
    tools=[SerperDevTool()],
    verbose=True,
    llm=llm1,
)

In [24]:
action_task = Task(
    name="ActionTask",
    description="Recommends a popular action movie",
    expected_output="A list of 10 popular movies",
    agent=movie_agent,
)
comedy_task = Task(
    name="ComedyTask",
    description="Recommends a popular comedy movie",
    expected_output="A list of 10 popular movies",
    agent=movie_agent,
)
drama_task = Task(
    name="DramaTask",
    description="Recommends a popular drama movie",
    expected_output="A list of 10 popular movies",
    agent=movie_agent,
)
sci_fi_task = Task(
    name="SciFiTask",
    description="Recommends a sci-fi movie",
    expected_output="A list of 10 popular movies",
    agent=movie_agent,
)

In [25]:
action_crew = Crew(
    agents=[movie_agent],
    tasks=[action_task],
    verbose=True,
)
comedy_crew = Crew(agents=[movie_agent], tasks=[comedy_task], verbose=True)
drama_crew = Crew(agents=[movie_agent], tasks=[drama_task], verbose=True)
sci_fi_crew = Crew(agents=[movie_agent], tasks=[sci_fi_task], verbose=True)

2025-01-21 16:31:25,762 - 140454436486016 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed
2025-01-21 16:31:25,764 - 140454436486016 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed
2025-01-21 16:31:25,766 - 140454436486016 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed
2025-01-21 16:31:25,768 - 140454436486016 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


In [26]:
GENRES = ["action", "comedy", "drama", "sci-fi"]


class GenreState(BaseModel):
    genre: str = ""

In [27]:
class MovieRecommendationFlow(Flow[GenreState]):
    @start()
    def input_genre(self):
        genre = input("Enter a genre: ")
        print(f"Genre input received: {genre}")
        self.state.genre = genre
        return genre

    @router(input_genre)
    def route_to_crew(self):
        genre = self.state.genre
        if genre not in GENRES:
            raise ValueError(f"Invalid genre: {genre}")
        if genre == "action":
            return "action"
        elif genre == "comedy":
            return "comedy"
        elif genre == "drama":
            return "drama"
        elif genre == "sci-fi":
            return "sci-fi"

    @listen("action")
    def action_movies(self, genre):
        recommendations = action_crew.kickoff()
        return recommendations

    @listen("comedy")
    def comedy_movies(self, genre):
        recommendations = comedy_crew.kickoff()
        return recommendations

    @listen("drama")
    def drama_movies(self, genre):
        recommendations = drama_crew.kickoff()
        return recommendations

    @listen("sci-fi")
    def sci_fi_movies(self, genre):
        recommendations = sci_fi_crew.kickoff()
        return recommendations

    @listen(or_("action_movies", "comedy_movies", "drama_movies", "sci_fi_movies"))
    def finalize_recommendation(self, recommendations):
        print("Final movie recommendations:")
        return recommendations

In [28]:
flow = MovieRecommendationFlow()
flow.plot()

Plot saved as crewai_flow.html


In [30]:
recommendations = await flow.kickoff_async()

Genre input received: action
# Agent: Recommend popular movie specific to the genre
## Task: Recommends a popular action movie
 

I encountered an error while trying to use the tool. This was the error: 'SERPER_API_KEY'.
 Tool Search the internet with Serper accepts these inputs: Tool Name: Search the internet with Serper
Tool Arguments: {'search_query': {'description': 'Mandatory search query you want to use to search the internet', 'type': 'str'}}
Tool Description: A tool that can be used to search the internet with a search_query. Supports different search types: 'search' (default), 'news'



# Agent: Recommend popular movie specific to the genre
## Thought: I need to find a list of popular action movies to provide the user with 10 recommendations. I'll search for current popular action movies online.
## Using tool: Search the internet with Serper
## Tool Input: 
"{\"search_query\": \"popular action movies 2023\"}"
## Tool Output: 

I encountered an error while trying to use the too